In [ ]:
!pip install transformers torch torchvision pillow

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image
import requests
import torch
import json

model_name = "Qwen/Qwen2-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(model_name)
model = Qwen2VLForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16).eval()
print("Qwen2 VL cargado correctamente")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

Qwen2 VL cargado correctamente


In [ ]:
!pip install qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 29.0 MB/s eta 0:00:00


In [ ]:
from qwen_vl_utils import process_vision_info
import json
import torch
import re

def analizar_prenda(imagen_url, id_prenda):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": imagen_url},
                {"type": "text", "text": """Eres un experto en moda y accesorios. Analiza la imagen y detecta el artículo principal.

Los tipos posibles son: camisa, camiseta, pantalon, vestido, zapatos, corbata, chaqueta, sudadera, jersey, bañador, falda, abrigo, bolso, gorro, RELOJ, CINTURÓN, BUFANDA, GAFAS o CALZADO.
Si no encaja en ninguno, usa la categoría más precisa que consideres (ej: 'complemento').

Devuelve SOLO un JSON con este formato:
[
    {
        "tipo": "tipo de artículo",
        "color": "color predominante",
        "estampado": "liso/rayas/cuadros/etc",
        "formalidad": "formal/informal/deporte",
        "temporada": ["invierno", "primavera", "verano", "otoño"]
    }
]
Analiza solo lo que veas. No escribas nada más que el JSON."""}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=1024)

    respuesta = processor.batch_decode(output, skip_special_tokens=True)[0]
    texto = respuesta.split("assistant")[-1].strip().replace("```json", "").replace("```", "").strip()

    if not texto.endswith("]"):
        texto = texto[:texto.rfind("}") + 1] + "]"

    try:
        prendas = json.loads(texto)
    except json.JSONDecodeError:
        objetos = re.findall(r'\{[^{}]+\}', texto)
        prendas = [json.loads(o) for o in objetos]

    resultado = []
    for i, prenda in enumerate(prendas):
        prenda["id"] = f"{id_prenda}_{i+1}"
        prenda["imagen_path"] = imagen_url
        resultado.append(prenda)

    return resultado



**COMPROBACIÓN DE LA CARPETA**

In [ ]:
import os
from google.colab import drive

# 1. Aseguramos que el Drive esté bien montado
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

base_path = "/content/drive/MyDrive"
items = os.listdir(base_path)

print("🔍 ESCANEANDO TODAS LAS CARPETAS 'ARMARIO'...\n")
print("-" * 50)

encontrada_alguna_llena = False

for item in items:
    # Buscamos coincidencias (mayúsculas o minúsculas)
    if "ARMARIO" in item.upper():
        ruta_completa = os.path.join(base_path, item)

        # Saltamos si no es una carpeta (por si hay un archivo que se llame así)
        if not os.path.isdir(ruta_completa):
            continue

        try:
            contenido = os.listdir(ruta_completa)
            # Filtramos solo imágenes para estar seguros
            fotos = [f for f in contenido if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]

            if len(fotos) > 0:
                print(f"✅ LLENA  -> '{item}'")
                print(f"   📂 Ruta: {ruta_completa}")
                print(f"   🖼️ Tiene {len(fotos)} imágenes.")
                print(f"   💡 COPIA ESTA RUTA PARA TU CÓDIGO.")
                encontrada_alguna_llena = True
            else:
                print(f"❌ VACÍA  -> '{item}'")
                print(f"   📂 Ruta: {ruta_completa}")
                print(f"   🚫 No tiene imágenes (total archivos: {len(contenido)})")

        except Exception as e:
            print(f"⚠️ ERROR  -> '{item}': No tengo permiso para leerla.")

        print("-" * 50)

if not encontrada_alguna_llena:
    print("\n🧐 ¡OJO! No he encontrado ninguna carpeta 'ARMARIO' con fotos.")
    print("Si en la web de Drive las ves, prueba a mover las fotos a una CARPETA NUEVA")
    print("creada por ti mismo, y llámala 'ARMARIO_NUEVO'.")

🔍 ESCANEANDO TODAS LAS CARPETAS 'ARMARIO'...

--------------------------------------------------
✅ LLENA  -> 'ARMARIO'
   📂 Ruta: /content/drive/MyDrive/ARMARIO
   🖼️ Tiene 22 imágenes.
   💡 COPIA ESTA RUTA PARA TU CÓDIGO.
--------------------------------------------------


In [ ]:
import json
import os
from pathlib import Path
from google.colab import drive

# 1. Asegurar que Drive está montado
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive', force_remount=True)

# 2. Rutas (ajustadas a lo que detectamos)
CARPETA_ROPA = "/content/drive/MyDrive/ARMARIO"
RUTA_JSON_DRIVE = "/content/drive/MyDrive/ARMARIO/armario.json"

# 3. Cargar armario existente (buscando en DRIVE, no en local)
if os.path.exists(RUTA_JSON_DRIVE):
    with open(RUTA_JSON_DRIVE, "r", encoding="utf-8") as f:
        armario = json.load(f)
    # Comprobamos por el nombre del archivo (ej: foto1.jpg) para que sea infalible
    rutas_existentes = {os.path.basename(p["imagen_path"]) for p in armario}
    print(f"Armario existente con {len(armario)} prendas")
else:
    armario = []
    rutas_existentes = set()
    print("Armario nuevo")

# 4. Recorrer imágenes
extensiones = {".jpg", ".jpeg", ".png", ".webp"}
path_objeto = Path(CARPETA_ROPA)

if not path_objeto.exists():
    print(f"❌ Error: No se encuentra la carpeta {CARPETA_ROPA}")
else:
    # Listamos los archivos de la carpeta
    imagenes = [p for p in path_objeto.iterdir() if p.suffix.lower() in extensiones]
    print(f"Imágenes encontradas en carpeta: {len(imagenes)}")

    for i, path_foto in enumerate(imagenes):
        ruta_completa = str(path_foto)
        nombre_archivo = path_foto.name # Solo el nombre del archivo

        # COMPROBACIÓN: Si el nombre ya está en el JSON, saltamos
        if nombre_archivo in rutas_existentes:
            print(f"⏭ Ya analizada: {nombre_archivo}")
            continue

        print(f"Analizando {nombre_archivo}...")
        try:
            # Tu función original
            resultado = analizar_prenda(ruta_completa, f"prenda_{len(armario)+1:03d}")
            for item in resultado:
                item["imagen_path"] = ruta_completa
                armario.append(item)
                print(f"✓ {item['id']} — {item['tipo']} {item['color']}")
        except Exception as e:
            print(f"✗ Error: {e}")

    # 5. Guardar el resultado tanto en el DRIVE como en LOCAL
    with open(RUTA_JSON_DRIVE, "w", encoding="utf-8") as f:
        json.dump(armario, f, indent=2, ensure_ascii=False)

    with open("armario.json", "w", encoding="utf-8") as f:
        json.dump(armario, f, indent=2, ensure_ascii=False)

    print(f"\nArmario guardado con {len(armario)} prendas")

Armario existente con 18 prendas
Imágenes encontradas en carpeta: 22
⏭ Ya analizada: abrigo.jpg
⏭ Ya analizada: polo.webp
⏭ Ya analizada: pantalontraje.png
⏭ Ya analizada: camisa.jpg
⏭ Ya analizada: jersey.jpg
⏭ Ya analizada: sudaderanegra.jpg
⏭ Ya analizada: mocasines.jpg
⏭ Ya analizada: bambas.jpg
⏭ Ya analizada: running.jpg
⏭ Ya analizada: sudadera.jpg
⏭ Ya analizada: zapatillas.jpg
⏭ Ya analizada: nikerojas.jpg
⏭ Ya analizada: bomber.jpg
⏭ Ya analizada: vaqueroscuro.jpg
⏭ Ya analizada: vaquuero.jpg
⏭ Ya analizada: camisetablanca.jpg
⏭ Ya analizada: vermudas.jpg
⏭ Ya analizada: sandalias.jpg
Analizando cinturon.jpg...
✓ prenda_019_1 — cinturón marrón oscuro
Analizando bufanda.jpg...
✓ prenda_020_1 —  Bufanda gris
Analizando reloj.jpg...
✓ prenda_021_1 — reloj negro
Analizando chandal.jpg...
✓ prenda_022_1 — pantalón gris

Armario guardado con 22 prendas


In [ ]:
import shutil
shutil.copy("armario.json", "/content/drive/MyDrive/ARMARIO/armario.json")
print("Armario guardado en Drive")

Armario guardado en Drive
